# RoadGuard AI — Notebook 03: GPU YOLO Model Training & Checkpoint Export

**Objective:** Fine-tune a lightweight pretrained Ultralytics YOLO detector (`yolo26n.pt` / `yolo11n.pt` / `yolov8n.pt`) on the prepared 4-class RDD2022 training/validation splits using domain-valid road augmentations, early stopping, checkpoint persistence, and Google Drive artifact export.

**Important:** Only `train` (70%) and `val` (15%) splits are accessed during training and model selection. The 15% `test` split remains untouched until `04_model_evaluation.ipynb`.

In [ ]:
%pip -q install ultralytics pyyaml pandas matplotlib

import json, shutil
from pathlib import Path
import torch, yaml
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

assert torch.cuda.is_available(), 'Enable an NVIDIA GPU in Colab (Runtime -> Change runtime type -> T4/L4/A100 GPU) before training!'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/RoadGuardAI')
(DRIVE_ROOT / 'models').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'experiments').mkdir(parents=True, exist_ok=True)

## 1. Configure Model Architecture, Hyperparameters & Road-Valid Augmentations
- **Horizontal Flip (`fliplr=0.5`)**: Valid across symmetric roadway lanes.
- **Vertical Flip (`flipud=0.0`)**: Disabled — inverted perspective road scenes are physically unrealistic.
- **Bounded Rotation (`degrees=5.0`)**: Prevents turning longitudinal (`D00`) cracks into transverse (`D10`) cracks.

In [ ]:
DATASET_YAML = Path('/content/roadguard_work/rdd2022_yolo_4class/dataset_config.yaml')
assert DATASET_YAML.is_file(), f'Run 01_dataset_preparation.ipynb first! Missing {DATASET_YAML}'

MODEL_ARCH = 'yolo26n.pt'   # Configurable: 'yolo26n.pt', 'yolo11n.pt', 'yolov8n.pt', 'yolo11s.pt'
EPOCHS = 50
BATCH_SIZE = 16
IMAGE_SIZE = 640
PATIENCE = 12
RESUME = False              # Set True to resume from Drive last.pt after a Colab disconnect

if RESUME and (DRIVE_ROOT / 'models' / 'last.pt').is_file():
    model = YOLO(str(DRIVE_ROOT / 'models' / 'last.pt'))
    resolved_arch = 'resumed_last.pt'
else:
    try:
        model = YOLO(MODEL_ARCH)
        resolved_arch = MODEL_ARCH
    except Exception:
        model = YOLO('yolo11n.pt')
        resolved_arch = 'yolo11n.pt'

print('Initialized baseline architecture:', resolved_arch)

## 2. Execute GPU Transfer-Learning Training
Checkpoints are written directly to Google Drive (`DRIVE_ROOT / 'experiments'`) so progress survives Colab session timeouts.

In [ ]:
results = model.train(
    data=str(DATASET_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    device=0,
    seed=42,
    project=str(DRIVE_ROOT / 'experiments'),
    name='rdd2022_baseline',
    exist_ok=True,
    resume=RESUME,
    hsv_h=0.012,
    hsv_s=0.45,
    hsv_v=0.35,
    degrees=5.0,
    translate=0.10,
    scale=0.35,
    flipud=0.0,
    fliplr=0.50,
    mosaic=1.0,
    close_mosaic=10,
)

run_dir = Path(results.save_dir)
best_pt = run_dir / 'weights' / 'best.pt'
last_pt = run_dir / 'weights' / 'last.pt'
if best_pt.is_file():
    shutil.copy2(best_pt, DRIVE_ROOT / 'models' / 'best.pt')
if last_pt.is_file():
    shutil.copy2(last_pt, DRIVE_ROOT / 'models' / 'last.pt')
print('Saved best checkpoint to:', DRIVE_ROOT / 'models' / 'best.pt')

## 3. Visualize Training & Validation Loss Curves and Download Checkpoint
Inspect convergence behavior on `results.csv` and optionally download `best.pt` directly to your local machine (`RoadGuardAI/models/best.pt`).

In [ ]:
hist_df = pd.read_csv(run_dir / 'results.csv')
hist_df.columns = [c.strip() for c in hist_df.columns]
display(hist_df.tail(5))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), dpi=130)
axes[0].plot(hist_df['epoch'], hist_df['train/box_loss'], label='Train Box Loss')
axes[0].plot(hist_df['epoch'], hist_df['val/box_loss'], label='Val Box Loss', linestyle='--')
axes[0].set_title('Bounding-Box Regression Loss')
axes[0].legend()

axes[1].plot(hist_df['epoch'], hist_df['metrics/mAP50(B)'], label='Val mAP@50', color='green')
axes[1].plot(hist_df['epoch'], hist_df['metrics/mAP50-95(B)'], label='Val mAP@50-95', color='purple')
axes[1].set_title('Validation mAP Progression')
axes[1].legend()
plt.tight_layout()
plt.show()

# Uncomment to trigger direct browser download of best.pt:
# from google.colab import files
# files.download(str(DRIVE_ROOT / 'models' / 'best.pt'))

### Q&A
- **How do I load the trained checkpoint into the local Windows web application?** Download `MyDrive/RoadGuardAI/models/best.pt`, place it at `RoadGuardAI/models/best.pt`, and click **Hot-Reload Checkpoint** in the RoadGuard AI web dashboard (no server restart needed).

### Data Analysis Key Findings
- Trained the 4-class YOLO road damage detector on GPU with perspective-valid augmentations (`flipud=0.0`, `degrees=5.0`), early stopping, and persistent Drive checkpointing.

### Insights or Next Steps
- Proceed to `04_model_evaluation.ipynb` to evaluate `best.pt` on the held-out 15% `test` split and run Classical-IP-vs-Detection experiments.